# 04 — Full pipeline demo video
detect → track → pose → classify → triage overlay on a **held-out** Okutama
test video, with the fine-tuned weights from notebooks 01–03.


### Setup (every notebook starts with this)
1. Runtime → Change runtime type → **T4 GPU** (free tier).
2. Zip your local `Project/` folder's `src/` and `scripts/` dirs as `src.zip`
   (`cd Project && zip -r src.zip src scripts`), then either upload it below
   or put it in Drive and adjust `SRC_ZIP`.


In [ ]:
# --- environment ---
!pip -q install ultralytics rtmlib onnxruntime-gpu
import torch, os
print('cuda:', torch.cuda.is_available())

# --- project code: upload src.zip (or mount Drive and set SRC_ZIP) ---
from pathlib import Path
SRC_ZIP = None  # e.g. '/content/drive/MyDrive/sar_project/src.zip'
if SRC_ZIP is None:
    from google.colab import files
    up = files.upload()  # choose src.zip
    SRC_ZIP = next(iter(up))
!mkdir -p /content/project && unzip -q -o "$SRC_ZIP" -d /content/project
import sys
sys.path.insert(0, '/content/project/src')
sys.path.insert(0, '/content/project')
print('project code ready')

# --- results go to Drive so they survive the session ---
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/sar_project_results'); OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Okutama-Action direct downloads (public Dropbox folder, verified working).
# preview=<file>&dl=1 selects a single file from the shared folder.
OKUTAMA_BASE = ('https://www.dropbox.com/scl/fo/9qvpsb3fsamvqzsa12149/'
                'APTyV-f01XLnJ0WFpZSBLOE?preview={name}&rlkey=7u7131amaul29amyr4jbnnu03&dl=1')

def fetch_okutama(name, dest='/content/data/okutama'):
    import subprocess, pathlib
    d = pathlib.Path(dest); d.mkdir(parents=True, exist_ok=True)
    zp = d / name
    if not zp.exists():
        subprocess.run(['curl', '-L', '-o', str(zp), OKUTAMA_BASE.format(name=name)], check=True)
    subprocess.run(['unzip', '-q', '-o', str(zp), '-d', str(d)], check=True)
    return d


In [ ]:
fetch_okutama('TestSetVideos.zip')   # held-out footage the models never saw
import glob
test_videos = sorted(glob.glob('/content/data/okutama/**/*.mov', recursive=True))
print(test_videos[:5])


In [ ]:
import config
config.MODELS_DIR = OUT   # picks up pose_mlp.pt trained in notebook 03
from render_demo import render
det_weights = str(OUT/'yolo11s_visdrone_best.pt')  # from notebook 01 (or 'yolo11s.pt')
out, n = render(test_videos[0], '/content/demo_raw.mp4', weights=det_weights,
                imgsz=1280, max_frames=1800, device=0, pose_device='cuda',
                caption='EECS 4422 — SAR triage pipeline (held-out footage)')


In [ ]:
# h264 re-encode so it plays everywhere, then keep in Drive
!ffmpeg -y -loglevel error -i /content/demo_raw.mp4 -c:v libx264 -pix_fmt yuv420p {OUT}/demo_final.mp4
print('saved to Drive:', OUT/'demo_final.mp4')
